# Kaggle submission

In [17]:
import os

import cv2
import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import train_aug_25d
import train_dino_pernorm_25d
import train_pernorm_25d

In [18]:
MODEL = 'dino'

if MODEL == 'segformer':
    train_module = train_aug_25d
    RUN_DIR = 'results/1794532-segformer_train_aug_25d'
elif MODEL == 'segformer_pernorm':
    train_module = train_pernorm_25d
    RUN_DIR = 'results/1807810-segformer_train_pernorm_25d'
elif MODEL == 'dino':
    train_module = train_dino_pernorm_25d
    RUN_DIR = 'results/1878954-dino_unet_train_dino_pernorm_25d'

DATA_DIR = './hw3_dataset'
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')
SUBMISSION_PATH = f'submission_{MODEL}.csv'

ENCODER = 'resnet34'
DINO_ENCODER = 'vit_small_patch16_dinov3.lvd1689m'
N_FOLDS = 5
THRESHOLD = 0.5

CLASSES = train_module.CLASSES
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(MODEL, device, CLASSES)

dino cuda ['stomach', 'small_bowel', 'large_bowel']


In [19]:
def rle_encode(img):
    """img: numpy array, 1 - mask, 0 - background. Returns run length as string formated."""
    pixels = img.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

In [20]:
def parse_id(id_str):
    parts = id_str.split('_')
    case = int(parts[0].replace('case', ''))
    day = int(parts[1].replace('day', ''))
    slice_id = int(parts[3])
    return case, day, slice_id


def get_test_df(path):
    df = pd.read_csv(path)

    df[['case', 'day', 'slice']] = df['id'].apply(lambda x: pd.Series(parse_id(x)))
    test_df = df[['case', 'day', 'slice']].drop_duplicates().reset_index(drop=True)

    test_df['image_path'] = test_df.apply(
        lambda r: f"{DATA_DIR}/test/case{r['case']}/day{r['day']}/slice_{int(r['slice'])}.png",
        axis=1,
    )

    return test_df.sort_values(['case', 'day', 'slice']).reset_index(drop=True)


test_df = get_test_df(path=SAMPLE_SUB_PATH)
test_df

,case,day,slice,image_path
0,1,0,1,./hw3_dataset/test/case1/day0/slice_1.png
1,1,0,2,./hw3_dataset/test/case1/day0/slice_2.png
2,1,0,3,./hw3_dataset/test/case1/day0/slice_3.png
3,1,0,4,./hw3_dataset/test/case1/day0/slice_4.png
4,1,0,5,./hw3_dataset/test/case1/day0/slice_5.png
...,...,...,...,...
8779,61,15,140,./hw3_dataset/test/case61/day15/slice_140.png
8780,61,15,141,./hw3_dataset/test/case61/day15/slice_141.png
8781,61,15,142,./hw3_dataset/test/case61/day15/slice_142.png
8782,61,15,143,./hw3_dataset/test/case61/day15/slice_143.png


In [21]:
class TestDataset(Dataset):
    def __init__(self, df, preprocess_input, scale_intensity, pad_fn):
        self.df = df
        self.path_of = {(row['case'], row['day'], row['slice']): row['image_path']
                        for _, row in df.iterrows()}
        self.preprocess_input = preprocess_input
        self.scale_intensity = scale_intensity
        self.pad_fn = pad_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        case, day, sl = row['case'], row['day'], row['slice']

        previous_key = (case, day, sl - 1) if (case, day, sl - 1) in self.path_of else (case, day, sl)
        next_key = (case, day, sl + 1) if (case, day, sl + 1) in self.path_of else (case, day, sl)

        slices = []
        for key in [previous_key, (case, day, sl), next_key]:
            image = cv2.imread(self.path_of[key], cv2.IMREAD_UNCHANGED)
            slices.append(self.scale_intensity(image.astype(np.float32)))
        image_3ch = np.stack(slices, axis=-1)

        norm_image = torch.from_numpy(self.preprocess_input(image_3ch).astype(np.float32)).permute(2, 0, 1)
        padded_image = self.pad_fn(norm_image).as_tensor()

        return padded_image, case, day, sl

`scale_intensity` has to use the p1/p99 the models were trained with

In [22]:
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
train_df['image_path'] = (f'{DATA_DIR}/train/case' + train_df['case'].astype(str)
                          + '/day' + train_df['day'].astype(int).astype(str)
                          + '/slice_' + train_df['slice'].astype(int).astype(str) + '.png')

preprocess_input, scale_intensity, pad = train_module.build_transforms(
    train_df, ENCODER, sample_size=500, seed=42
)

test_dataset = TestDataset(test_df, preprocess_input, scale_intensity, pad)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

image, case, day, sl = test_dataset[0]
print(tuple(image.shape), f'[{image.min():.2f}, {image.max():.2f}]', case, day, sl)

(3, 288, 288) [-2.12, 2.64] 1 0 1


Cross-validation trained five models, one per fold. So let's make their ensemble.

In [23]:
models = []
for fold in range(N_FOLDS):
    if MODEL == 'dino':
        model = train_dino_pernorm_25d.DinoUnet(DINO_ENCODER, classes=len(CLASSES), freeze_encoder=False)
    else:
        model = smp.create_model('segformer', encoder_name=ENCODER, encoder_weights=None,
                                 classes=len(CLASSES))

    checkpoint = torch.load(f'{RUN_DIR}/fold{fold}/checkpoint_best.pt', map_location=device,
                            weights_only=False)
    model.load_state_dict(checkpoint['model_state'])
    models.append(model.to(device).eval())
    print(f"fold {fold}: epoch {checkpoint['epoch']}, val dice {checkpoint['best_val_dice']:.4f}")

fold 0: epoch 11, val dice 0.6947
fold 1: epoch 18, val dice 0.7519
fold 2: epoch 22, val dice 0.7771
fold 3: epoch 42, val dice 0.6298
fold 4: epoch 48, val dice 0.6217


In [24]:
rows = []

with torch.no_grad():
    for images, cases, days, slices in tqdm(test_loader, desc='Predicting'):
        images = images.to(device)

        probabilities = torch.zeros(images.shape[0], len(CLASSES), images.shape[2], images.shape[3],
                                    device=device)
        for model in models:
            probabilities += torch.sigmoid(model(images))
        probabilities /= len(models)

        # undo DivisiblePad: the submitted mask must be at the original image size
        top = (images.shape[2] - train_module.MASK_H) // 2
        left = (images.shape[3] - train_module.MASK_W) // 2
        masks = (probabilities > THRESHOLD).to(torch.uint8).cpu().numpy()
        masks = masks[:, :, top:top + train_module.MASK_H, left:left + train_module.MASK_W]

        for i in range(len(masks)):
            slice_id = f'case{int(cases[i])}_day{int(days[i])}_slice_{int(slices[i]):04d}'
            for c, cls in enumerate(CLASSES):
                rows.append({'id': f'{slice_id}_class_{cls}',
                             'segmentation': rle_encode(masks[i, c])})

submission_df = pd.DataFrame(rows)
len(submission_df)

Predicting:   0%|          | 0/549 [00:00<?, ?it/s]

Predicting: 100%|██████████| 549/549 [05:25<00:00,  1.69it/s]


26352

In [25]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
expected_ids = sample_sub['id'] + '_class_' + sample_sub['class']

assert submission_df['id'].is_unique, 'duplicate ids'
assert set(submission_df['id']) == set(expected_ids), 'ids do not match the sample submission'

submission_df.to_csv(SUBMISSION_PATH, index=False)

print(SUBMISSION_PATH, (submission_df['segmentation'] != '').sum(), 'non-empty masks out of',
      len(submission_df))
submission_df.head()

submission_dino.csv 8209 non-empty masks out of 26352


,id,segmentation
0,case1_day0_slice_0001_class_stomach,
1,case1_day0_slice_0001_class_small_bowel,
2,case1_day0_slice_0001_class_large_bowel,
3,case1_day0_slice_0002_class_stomach,
4,case1_day0_slice_0002_class_small_bowel,
